# ECOSTRESS mineral library — `HyFourier` maps by instrument

Loads mineral spectra from the JPL ECOSTRESS spectral library (ASTER format), converts
wavelengths from **micrometers → nm** and reflectance from **percent → fraction**, assigns
**mineral-family groups** using the same `mineral_families.csv` mapping as
`usgs_fourier_library.ipynb`, builds one **`HyFourier` map per instrument/wavelength grid** in a
**`FourierArchive`**, and saves `public/libraries/ecostress_minerals.fda` for iSpec.

In [1]:
import glob
import os
import re
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

import hylite
from hylite import HyLibrary
from hylite.analyse.fourier import HyFourier, FourierArchive, FOURIER_ARCHIVE_EXTENSION

ECOSTRESS_DIR = '/Users/thiele67/Documents/data/Libraries/Spectra ECOSTRESS/Minerals'
FAMILIES_CSV = '/Users/thiele67/Documents/data/Libraries/Spectra USGS/mineral_families.csv'
REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(hylite.__file__), '..', '..', 'ispec'))
OUT_DIR = os.path.join(REPO_ROOT, 'public', 'libraries')
FDA_PATH = os.path.join(OUT_DIR, 'ecostress_minerals')
os.makedirs(OUT_DIR, exist_ok=True)

ARCHIVE_PREFIX = 'ecostress_minerals'
OWNER_INSTRUMENT_SUFFIX = re.compile(
    r'\.(?:jhu|jpl|usgs|nmnh[\w-]*)\.(?:beckman|perkin|nicolet|perknic)(?:\.spectrum)?$',
    re.IGNORECASE,
)

CLASS_TO_FAMILY = {
    'carbonate': 'carbonate',
    'sulfate': 'sulphate',
    'sulfide': 'sulphide',
    'oxide': 'oxide',
    'phosphate': 'phosphate',
    'hydroxide': 'hydroxide',
    'element': 'element',
}

## 1. Mineral families (shared with USGS library)


In [3]:
def load_mineral_families(path):
    families = {}
    family_names = set()
    with open(path, 'r', encoding='utf-8') as handle:
        for line in handle:
            line = line.strip()
            if not line or ',' not in line:
                continue
            mineral, family = line.split(',', 1)
            mineral = mineral.strip().lower()
            family = family.strip().lower()
            families[mineral] = family
            family_names.add(family)
    return families, family_names


def mineral_from_name(name_field):
    """Extract mineral name from ECOSTRESS `Name:` header (e.g. 'Mimetite Pb_5...')."""
    token = re.split(r'[\s,(]+', name_field.strip())[0]
    return re.sub(r'[^a-zA-Z-]', '', token).lower()


def resolve_family(mineral, mineral_class, families, family_names):
    if mineral in families:
        return families[mineral]
    if mineral in family_names:
        return mineral
    return CLASS_TO_FAMILY.get(mineral_class.strip().lower(), 'unclassified')


families, family_names = load_mineral_families(FAMILIES_CSV)
print(f'Loaded {len(families)} mineral → family mappings')

Loaded 212 mineral → family mappings


## 2. Load ECOSTRESS ASCII spectra

Each `.spectrum.txt` file has a metadata header (`X Units: Wavelength (micrometers)`,
`Y Units: Reflectance (percent)` or `Transmittance (percent)`) followed by tab-separated pairs.
Transmission/transmittance spectra are stored as **R = 1 − T** (fraction) for comparability with
reflectance measurements.

In [4]:
def is_transmission(meta):
    measurement = meta.get('Measurement', '').lower()
    y_units = meta.get('Y Units', '').lower()
    return 'transmis' in measurement or 'transmis' in y_units


def y_to_reflectance(meta, y_fraction):
    """Convert percent/fraction Y to reflectance-like scale for library use."""
    if is_transmission(meta):
        return 1.0 - y_fraction
    return y_fraction


def simplify_spectrum_label(path):
    stem = os.path.basename(path)
    if stem.endswith('.spectrum.txt'):
        stem = stem[: -len('.spectrum.txt')]
    elif stem.endswith('.txt'):
        stem = os.path.splitext(stem)[0]
    if stem.startswith('mineral.'):
        stem = stem[len('mineral.') :]
    stem = OWNER_INSTRUMENT_SUFFIX.sub('', stem)
    return stem


def parse_ecostress_spectrum(path, families, family_names):
    meta = {}
    rows = []
    with open(path, 'r', encoding='utf-8', errors='replace') as handle:
        for line in handle:
            stripped = line.strip()
            if not stripped:
                continue
            if ':' in stripped and not stripped[0].isdigit() and not stripped.startswith('-'):
                key, value = stripped.split(':', 1)
                meta[key.strip()] = value.strip()
                continue
            parts = stripped.split()
            if len(parts) != 2:
                continue
            try:
                rows.append((float(parts[0]), float(parts[1])))
            except ValueError:
                continue

    arr = np.asarray(rows, dtype=np.float64)
    order = np.argsort(arr[:, 0])
    wav_nm = arr[order, 0] * 1000.0
    y_frac = arr[order, 1] / 100.0
    refl = y_to_reflectance(meta, y_frac)
    refl[~np.isfinite(refl)] = np.nan

    mineral = mineral_from_name(meta.get('Name', os.path.basename(path)))
    family = resolve_family(mineral, meta.get('Class', ''), families, family_names)
    instrument = os.path.basename(path).replace('.spectrum.txt', '').split('.')[-1]

    return {
        'label': simplify_spectrum_label(path),
        'wav': wav_nm.astype(np.float32),
        'refl': refl.astype(np.float32),
        'mineral': mineral,
        'family': family,
        'instrument': instrument,
        'meta': meta,
    }


def align_spectrum(wav, refl, ref_wav):
    if len(wav) == len(ref_wav) and np.allclose(wav, ref_wav, atol=0.15, rtol=0):
        return refl.astype(np.float32)
    return np.interp(ref_wav, wav, refl, left=np.nan, right=np.nan).astype(np.float32)


def grid_signature(wav):
    return (len(wav), round(float(wav[0]), 1), round(float(wav[-1]), 1))


def archive_key_map(by_grid):
    by_inst = defaultdict(set)
    for inst, sig in by_grid:
        by_inst[inst].add(sig)

    mapping = {}
    for inst in sorted(by_inst):
        sigs = sorted(by_inst[inst], key=lambda s: (s[1], s[2]))
        for index, sig in enumerate(sigs):
            if len(sigs) == 1:
                mapping[(inst, sig)] = f'{ARCHIVE_PREFIX}:{inst}'
            else:
                mapping[(inst, sig)] = f'{ARCHIVE_PREFIX}:{inst}{index + 1}'
    return mapping


paths = sorted(glob.glob(os.path.join(ECOSTRESS_DIR, '*.spectrum.txt')))
by_grid = defaultdict(list)
errors = []
transmission_count = 0
for path in paths:
    try:
        entry = parse_ecostress_spectrum(path, families, family_names)
        if is_transmission(entry['meta']):
            transmission_count += 1
        by_grid[(entry['instrument'], grid_signature(entry['wav']))].append(entry)
    except Exception as exc:
        errors.append((path, exc))

archive_keys = archive_key_map(by_grid)

print(f'Found {len(paths)} spectrum files → {sum(len(v) for v in by_grid.values())} loaded')
print(f'Converted {transmission_count} transmission/transmittance spectra with R = 1 - T')
if errors:
    print(f'WARNING: skipped {len(errors)} files')
for (inst, sig), key in sorted(archive_keys.items(), key=lambda item: (item[0][0], item[0][1][1])):
    entries = by_grid[(inst, sig)]
    entry = entries[0]
    print(
        f"  {key:32s}  n={len(entry['wav']):4d}  "
        f"{entry['wav'][0]:8.1f}-{entry['wav'][-1]:8.1f} nm  "
        f"{len(entries):4d} spectra"
    )

Found 1609 spectrum files → 1609 loaded
  beckman                       n= 826     400.0-  2500.0 nm   430 spectra
  perkin                        n=2101     400.0-  2500.0 nm   406 spectra
  perknic_2752_400.0_13900.0    n=2752     400.0- 13900.0 nm    17 spectra
  perknic_2231_400.0_14051.0    n=2231     400.0- 14051.0 nm     4 spectra
  nicolet_2256_2000.3_15385.3   n=2256    2000.3- 15385.3 nm   403 spectra
  nicolet_2287_2079.5_25044.2   n=2287    2079.5- 25044.2 nm   328 spectra
  nicolet_2287_2079.5_25044.0   n=2287    2079.5- 25044.0 nm    21 spectra


## 3. Build `HyLibrary` groups and `HyFourier` maps per instrument


In [5]:
def feature_wavelength_range(wav):
    lo = max(400.0, float(wav[0]))
    hi = min(15000.0, float(wav[-1]))
    return lo, hi


archive = FourierArchive()
for (inst, sig), archive_key in sorted(archive_keys.items(), key=lambda item: (item[0][0], item[0][1][1])):
    entries = by_grid[(inst, sig)]
    ref_wav = entries[0]['wav']
    for entry in entries[1:]:
        if not (
            len(entry['wav']) == len(ref_wav)
            and np.allclose(entry['wav'], ref_wav, atol=0.15, rtol=0)
        ):
            entry['refl'] = align_spectrum(entry['wav'], entry['refl'], ref_wav)
            entry['wav'] = ref_wav

    names = [entry['label'] for entry in entries]
    spectra = np.stack([entry['refl'] for entry in entries], axis=0)[:, None, :]
    lib = HyLibrary(spectra, lab=names, wav=ref_wav)

    group_ids = defaultdict(list)
    for index, entry in enumerate(entries):
        group_ids[entry['family']].append(index)
    for family, ids in sorted(group_ids.items()):
        lib.add_group(family, ids)

    lo, hi = feature_wavelength_range(ref_wav)
    subset = lib.export_bands((lo, hi))
    hf = HyFourier(subset, padding='cosine', max_freq=0.25, vb=True)
    hf.precomputeExtrema(kde_sigma=10.0, vb=True)
    archive[archive_key] = hf

    print(
        f"{archive_key:32s}  {lib.sample_count():4d} samples  "
        f"{len(lib.get_groups()):2d} groups  HyFourier ({lo:.0f}-{hi:.0f} nm)"
    )

beckman                        430 samples  34 groups  HyFourier (400-2500 nm)


perkin                         406 samples  33 groups  HyFourier (400-2500 nm)


perknic_2752_400.0_13900.0      17 samples   3 groups  HyFourier (400-13900 nm)


perknic_2231_400.0_14051.0       4 samples   3 groups  HyFourier (400-14051 nm)


nicolet_2256_2000.3_15385.3    403 samples  33 groups  HyFourier (2000-15000 nm)


nicolet_2287_2079.5_25044.2    328 samples  29 groups  HyFourier (2080-15000 nm)


nicolet_2287_2079.5_25044.0     21 samples   6 groups  HyFourier (2080-15000 nm)


## 4. Save archive and verify


In [6]:
archive.save(FDA_PATH)
print('Saved', FDA_PATH + FOURIER_ARCHIVE_EXTENSION)

loaded = FourierArchive.load(FDA_PATH)
print('Archive keys:', list(loaded.keys()))

Saved /Users/thiele67/Documents/python/ispec/public/libraries/ecostress_minerals.fda
Archive keys: ['beckman', 'perkin', 'perknic_2752_400.0_13900.0', 'perknic_2231_400.0_14051.0', 'nicolet_2256_2000.3_15385.3', 'nicolet_2287_2079.5_25044.2', 'nicolet_2287_2079.5_25044.0']


In [ ]:
# Quick visual check — top 2200 nm match on beckman grid
names, scores = loaded.search('2200', confidence=20.0, n_result=1)
spectra = loaded.getSpectra(names[0])
plt.figure(figsize=(8, 3))
plt.plot(spectra.get_wavelengths(), spectra.data[0, 0], lw=0.8)
plt.axvline(2200, color='r', alpha=0.4)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Reflectance')
plt.title(names[0])
plt.tight_layout()